In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, TimestampType, DateType

In [0]:
dbutils.widgets.text("bronze_catalog", "dbr_dev")
dbutils.widgets.text("bronze_schema", "artemzharkov10_bronze")

dbutils.widgets.text("silver_catalog", "dbr_dev")
dbutils.widgets.text("silver_schema", "artemzharkov10_silver")


BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

In [0]:
BRONZE_TABLE = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_historical_car_accidents_weather"

SILVER_PARENT_TABLE = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_historical_car_accidents_weather"
SILVER_TARGET_TABLE = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_historical_car_accidents_metrics_for_algorithm"


In [0]:
df_bronze_history_car = spark.table(BRONZE_TABLE)

df_silver_parent = (
    df_bronze_history_car
    .withColumn("lat", F.col("lat").cast(DoubleType())) 
    .withColumn("lon", F.col("lon").cast(DoubleType())) 
    .withColumn("target_acc_date", F.col("target_acc_date").cast(DateType())) 
    .withColumn("time", F.col("time").cast(TimestampType())) 
    .withColumn("temperature_2m", F.col("temperature_2m").cast(DoubleType()))
    .withColumn("precipitation", F.col("precipitation").cast(DoubleType()))
    .withColumn("wind_speed_10m", F.col("wind_speed_10m").cast(DoubleType()))
    .withColumn("soil_temp", F.col("soil_temp").cast(DoubleType()))
    .withColumn("humidity_2m", F.col("humidity_2m").cast(DoubleType()))
    .withColumn("dew_point_2m", F.col("dew_point_2m").cast(DoubleType()))
)

(df_silver_parent.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_PARENT_TABLE))

In [0]:
df_silver_target = (
    df_silver_parent
    .select(
        F.col("target_acc_date"),
        F.col("time"),
        F.col("lat"),
        F.col("lon"),
        F.col("precipitation"),
        F.col("soil_temp")
    )
)
(df_silver_target.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TARGET_TABLE))